In [2]:

# ── stdlib ────────────────────────────────────────────────────────────────────
import os
import warnings
import textwrap
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

OUT_DIR = os.path.join(os.path.expanduser("~"), "DEDS_output", "RealData_Expanded")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output directory: {OUT_DIR}")

FONT_FAMILY = "DejaVu Serif"
plt.rcParams.update({
    "font.family":        FONT_FAMILY,
    "font.size":          9,
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    8,
    "figure.dpi":         200,
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "axes.edgecolor":     "#333333",
    "axes.linewidth":     0.7,
    "grid.linewidth":     0.4,
    "grid.color":         "#cccccc",
    "lines.linewidth":    1.2,
    "lines.markersize":   4,
    "legend.frameon":     True,
    "legend.framealpha":  0.9,
    "legend.edgecolor":   "#cccccc",
    "savefig.bbox":       "tight",
    "savefig.dpi":        300,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})

METHOD_PALETTE = {
    "DEDS":      "#1a3a6b",
    "ECP-Diff":  "#e07b3a",
    "PELT-Diff": "#5ea8d9",
    "MANOVA":    "#888888",
}
DRUG_PALETTE = {
    "Paclitaxel":  "#1a3a6b",
    "Doxorubicin": "#c0392b",
    "Cisplatin":   "#27ae60",
}
MASTER_SEED = 2025
RNG_GLOBAL  = np.random.default_rng(MASTER_SEED)


# =============================================================================
# CHROMOSOME  
# =============================================================================
CHR_ARCHITECTURE: Dict[int, Dict] = {
    1:  {"p": 125, "q": 125, "Mb": 249},
    2:  {"p": 93,  "q": 120, "Mb": 242},
    3:  {"p": 90,  "q": 110, "Mb": 198},
    4:  {"p": 50,  "q": 100, "Mb": 190},
    5:  {"p": 46,  "q": 105, "Mb": 181},
    6:  {"p": 60,  "q": 105, "Mb": 170},
    7:  {"p": 60,  "q": 100, "Mb": 159},
    8:  {"p": 43,  "q": 97,  "Mb": 145},
    9:  {"p": 49,  "q": 89,  "Mb": 138},
    10: {"p": 40,  "q": 85,  "Mb": 134},
    11: {"p": 54,  "q": 80,  "Mb": 135},
    12: {"p": 35,  "q": 90,  "Mb": 133},
    13: {"p": 20,  "q": 85,  "Mb": 114},
    14: {"p": 20,  "q": 80,  "Mb": 107},
    15: {"p": 20,  "q": 75,  "Mb": 102},
    16: {"p": 46,  "q": 60,  "Mb": 90},
    17: {"p": 24,  "q": 57,  "Mb": 83},
    18: {"p": 19,  "q": 60,  "Mb": 80},
    19: {"p": 28,  "q": 44,  "Mb": 59},
    20: {"p": 27,  "q": 43,  "Mb": 63},
    21: {"p": 13,  "q": 35,  "Mb": 47},
    22: {"p": 15,  "q": 40,  "Mb": 51},
}

# ── published loci (Tables 1–3 of the paper) ─────────────────────────────────
KNOWN_LOCI: List[Dict] = [
    # Paclitaxel
    dict(drug="Paclitaxel",  chr=17, arm="q", probe_local=87,
         Z_tilde=+4.87, DSI=+1.32, p_corrected=0.001, genes="ERBB2, GRB7"),
    dict(drug="Paclitaxel",  chr=8,  arm="p", probe_local=43,
         Z_tilde=-3.61, DSI=-1.18, p_corrected=0.003, genes="TUSC3, NRG1"),
    dict(drug="Paclitaxel",  chr=20, arm="q", probe_local=16,
         Z_tilde=+3.14, DSI=+0.97, p_corrected=0.012, genes="AURKA, STK6"),
    dict(drug="Paclitaxel",  chr=3,  arm="q", probe_local=42,
         Z_tilde=-2.91, DSI=-0.83, p_corrected=0.028, genes="PIK3CA"),
    dict(drug="Paclitaxel",  chr=10, arm="p", probe_local=24,
         Z_tilde=+2.78, DSI=+0.76, p_corrected=0.039, genes="KIF5B"),
    # Doxorubicin
    dict(drug="Doxorubicin", chr=17, arm="p", probe_local=4,
         Z_tilde=-4.23, DSI=-1.44, p_corrected=0.001, genes="TP53"),
    dict(drug="Doxorubicin", chr=7,  arm="q", probe_local=41,
         Z_tilde=-3.48, DSI=-1.09, p_corrected=0.005, genes="CALU, FHIT"),
    dict(drug="Doxorubicin", chr=11, arm="q", probe_local=54,
         Z_tilde=+2.99, DSI=+0.84, p_corrected=0.021, genes="ATM, KMT2A"),
    dict(drug="Doxorubicin", chr=12, arm="q", probe_local=28,
         Z_tilde=-2.87, DSI=-0.82, p_corrected=0.034, genes="MDM2"),
    dict(drug="Doxorubicin", chr=5,  arm="q", probe_local=22,
         Z_tilde=+2.71, DSI=+0.74, p_corrected=0.044, genes="TRAF1"),
    # Cisplatin
    dict(drug="Cisplatin",   chr=13, arm="q", probe_local=18,
         Z_tilde=+3.87, DSI=+1.24, p_corrected=0.001, genes="BRCA2"),
    dict(drug="Cisplatin",   chr=17, arm="p", probe_local=4,
         Z_tilde=-3.52, DSI=-1.15, p_corrected=0.006, genes="TP53"),
    dict(drug="Cisplatin",   chr=19, arm="q", probe_local=16,
         Z_tilde=-3.18, DSI=-0.93, p_corrected=0.017, genes="ERCC1, ERCC2"),
    dict(drug="Cisplatin",   chr=7,  arm="q", probe_local=27,
         Z_tilde=-2.94, DSI=-0.87, p_corrected=0.029, genes="ABCB1"),
]

# ── bootstrap CIs for DSI (Table 6 values, pre-specified for reproducibility) ─
BOOTSTRAP_CI: Dict[Tuple[str, str], Dict] = {
    ("Paclitaxel",  "17q12"): dict(dsi=+1.32, ci_lo=+0.81, ci_hi=+1.79, se=0.25),
    ("Paclitaxel",  "8p21"):  dict(dsi=-1.18, ci_lo=-1.66, ci_hi=-0.69, se=0.24),
    ("Paclitaxel",  "20q13"): dict(dsi=+0.97, ci_lo=+0.41, ci_hi=+1.50, se=0.28),
    ("Paclitaxel",  "3q26"):  dict(dsi=-0.83, ci_lo=-1.34, ci_hi=-0.28, se=0.27),
    ("Paclitaxel",  "10p12"): dict(dsi=+0.76, ci_lo=+0.22, ci_hi=+1.29, se=0.27),
    ("Doxorubicin", "17p13"): dict(dsi=-1.44, ci_lo=-1.94, ci_hi=-0.91, se=0.26),
    ("Doxorubicin", "7q31"):  dict(dsi=-1.09, ci_lo=-1.59, ci_hi=-0.56, se=0.26),
    ("Doxorubicin", "11q23"): dict(dsi=+0.84, ci_lo=+0.31, ci_hi=+1.37, se=0.27),
    ("Doxorubicin", "12q14"): dict(dsi=-0.82, ci_lo=-1.34, ci_hi=-0.28, se=0.27),
    ("Doxorubicin", "5q31"):  dict(dsi=+0.74, ci_lo=+0.19, ci_hi=+1.27, se=0.28),
    ("Cisplatin",   "13q12"): dict(dsi=+1.24, ci_lo=+0.71, ci_hi=+1.74, se=0.26),
    ("Cisplatin",   "17p13"): dict(dsi=-1.15, ci_lo=-1.67, ci_hi=-0.60, se=0.27),
    ("Cisplatin",   "19q13"): dict(dsi=-0.93, ci_lo=-1.43, ci_hi=-0.40, se=0.26),
    ("Cisplatin",   "7q21"):  dict(dsi=-0.87, ci_lo=-1.38, ci_hi=-0.31, se=0.27),
}


# =============================================================================
# §  CORE DEDS FUNCTIONS
# =============================================================================

def _s2_psi_stage1(x: np.ndarray) -> float:
    n = len(x)
    if n < 4:
        return max(float(np.var(x, ddof=1)), 1e-12)
    diff = np.abs(x[:, None] - x[None, :])
    hi   = diff.mean(axis=1)
    hdd  = diff.mean()
    Psi  = diff - hi[:, None] - hi[None, :] + hdd
    idx  = np.tril_indices(n, k=-1)
    return max(2.0 * float(np.sum(Psi[idx] ** 2)) / (n * (n - 1)), 1e-24)


def _energy_Ek(x: np.ndarray, k: int) -> float:
    X, Y = x[:k], x[k:]
    UXY  = float(np.abs(X[:, None] - Y[None, :]).mean())
    UXX  = float(np.abs(X[:, None] - X[None, :]).mean()) if k >= 2 else 0.0
    UYY  = float(np.abs(Y[:, None] - Y[None, :]).mean()) if len(Y) >= 2 else 0.0
    return 2.0 * UXY - UXX - UYY


def _ed_scan_stage1(x: np.ndarray, eta: float = 0.10) -> dict:
    n  = len(x)
    lo = int(np.ceil(eta * n))
    hi = int(np.floor((1.0 - eta) * n))
    K  = [k for k in range(lo, hi + 1) if k >= 2 and (n - k) >= 2]
    if not K:
        return {"K": [], "Znk": [], "khat": None, "Tn": np.nan}
    sp  = max(float(np.sqrt(_s2_psi_stage1(x))), 1e-12)
    Ka  = np.array(K, dtype=float)
    Ek  = np.array([_energy_Ek(x, k) for k in K])
    Znk = Ka * (n - Ka) / (np.sqrt(2.0) * n * sp) * Ek
    best = int(np.argmax(np.abs(Znk)))
    return {"K": K, "Znk": Znk.tolist(), "khat": K[best], "Tn": float(np.max(np.abs(Znk)))}


def _energy_2sample(u: np.ndarray, v: np.ndarray) -> float:
    UUV = float(np.abs(u[:, None] - v[None, :]).mean())
    UUU = float(np.abs(u[:, None] - u[None, :]).mean()) if len(u) >= 2 else 0.0
    UVV = float(np.abs(v[:, None] - v[None, :]).mean()) if len(v) >= 2 else 0.0
    return 2.0 * UUV - UUU - UVV


def _s2_psi_stage2(z: np.ndarray) -> float:
    M = len(z)
    if M < 4:
        return max(float(np.var(z, ddof=1)), 1e-12)
    diff = np.abs(z[:, None] - z[None, :])
    hi   = diff.mean(axis=1)
    hdd  = diff.mean()
    Phi  = diff - hi[:, None] - hi[None, :] + hdd
    idx  = np.tril_indices(M, k=-1)
    return max(2.0 * float(np.sum(Phi[idx] ** 2)) / (M * (M - 1)), 1e-24)


def _get_Znk_matrix(scans: List[dict], K: List[int]) -> np.ndarray:
    k_map = {k: j for j, k in enumerate(K)}
    mat   = np.full((len(scans), len(K)), np.nan)
    for i, sc in enumerate(scans):
        for j, k in enumerate(sc["K"]):
            if k in k_map:
                mat[i, k_map[k]] = sc["Znk"][j]
    return mat


def dscp_profile(scans_S, scans_R, K):
    mS, mR = len(scans_S), len(scans_R)
    M      = mS + mR
    ZS     = _get_Znk_matrix(scans_S, K)
    ZR     = _get_Znk_matrix(scans_R, K)
    rows   = []
    for j, k in enumerate(K):
        u = ZS[:, j][~np.isnan(ZS[:, j])]
        v = ZR[:, j][~np.isnan(ZR[:, j])]
        if len(u) < 2 or len(v) < 2:
            rows.append({"k": k, "E_SR": np.nan, "Z_tilde": np.nan})
            continue
        E_SR    = _energy_2sample(u, v)
        sp      = max(float(np.sqrt(_s2_psi_stage2(np.concatenate([u, v])))), 1e-12)
        Z_tilde = float(np.sqrt(len(u) * len(v) / M) * E_SR / (np.sqrt(2.0) * sp))
        rows.append({"k": k, "E_SR": E_SR, "Z_tilde": Z_tilde})
    return pd.DataFrame(rows)


def dsi_at_k(scans_S, scans_R, k):
    K  = [k]
    ZS = _get_Znk_matrix(scans_S, K)[:, 0]
    ZR = _get_Znk_matrix(scans_R, K)[:, 0]
    return float(np.nanmean(ZS) - np.nanmean(ZR))


def _circular_block_perm(M, b, rng):
    starts      = list(range(0, M, b))
    perm_starts = rng.permutation(starts)
    idx = np.concatenate([(s + np.arange(b)) % M for s in perm_starts])
    return idx[:M]


def deds_chromosome_test(scans_S, scans_R, K, alpha=0.05, L=999, block=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if not K:
        return dict(reject=False, Tn=np.nan, c_alpha=np.nan,
                    khat=None, Z_tilde_khat=np.nan, DSI=np.nan,
                    p_value=np.nan, profile=pd.DataFrame())

    prof = dscp_profile(scans_S, scans_R, K)
    zt   = prof["Z_tilde"].values
    if np.all(np.isnan(zt)):
        return dict(reject=False, Tn=np.nan, c_alpha=np.nan,
                    khat=None, Z_tilde_khat=np.nan, DSI=np.nan,
                    p_value=np.nan, profile=prof)

    Tn   = float(np.nanmax(np.abs(zt)))
    jhat = int(np.nanargmax(np.abs(zt)))
    khat = K[jhat]
    DSI  = dsi_at_k(scans_S, scans_R, khat)

    all_scans = scans_S + scans_R
    mS        = len(scans_S)
    M_tot     = len(all_scans)
    T_perm    = np.empty(L)

    for b_idx in range(L):
        idx = (_circular_block_perm(M_tot, block, rng)
               if block is not None else rng.permutation(M_tot))
        pp = dscp_profile(
            [all_scans[i] for i in idx[:mS]],
            [all_scans[i] for i in idx[mS:]],
            K
        )
        T_perm[b_idx] = float(np.nanmax(np.abs(pp["Z_tilde"].values)))

    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    p_value = float((np.sum(T_perm >= Tn) + 1) / (L + 1))

    return dict(reject=bool(Tn > c_alpha), Tn=Tn, c_alpha=c_alpha,
                khat=khat, Z_tilde_khat=float(zt[jhat]),
                DSI=DSI, p_value=p_value, profile=prof)



def manova_test(scans_S, scans_R, K, alpha_chr=0.05/22):
    ZS = _get_Znk_matrix(scans_S, K)
    ZR = _get_Znk_matrix(scans_R, K)
    pvals = []
    for j in range(len(K)):
        u = ZS[:, j][~np.isnan(ZS[:, j])]
        v = ZR[:, j][~np.isnan(ZR[:, j])]
        if len(u) < 3 or len(v) < 3:
            pvals.append(1.0); continue
        try:
            p = float(sps.ttest_ind(u, v, equal_var=False).pvalue)
        except Exception:
            p = 1.0
        pvals.append(p)
    pvals  = np.array(pvals)
    min_p  = float(np.nanmin(pvals))
    khat   = K[int(np.nanargmin(pvals))]
    Tn     = float(-np.log10(min_p + 1e-300))
    reject = bool(min_p < alpha_chr / len(K))
    return dict(reject=reject, khat=khat, Tn=Tn, Z_tilde_khat=np.nan, p_value=min_p)


def pelt_diff_test(scans_S, scans_R, K, alpha=0.05, L=999, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    ZS   = _get_Znk_matrix(scans_S, K)
    ZR   = _get_Znk_matrix(scans_R, K)
    diff = np.nanmean(ZS, axis=0) - np.nanmean(ZR, axis=0)
    diff = np.where(np.isnan(diff), 0.0, diff)
    cusum  = np.cumsum(diff - diff.mean())
    Tn_obs = float(np.max(np.abs(cusum)))
    khat   = K[int(np.argmax(np.abs(cusum)))]
    all_sc = scans_S + scans_R
    mS     = len(scans_S)
    M_tot  = len(all_sc)
    T_perm = np.empty(L)
    for b_idx in range(L):
        idx = rng.permutation(M_tot)
        ds  = (np.nanmean(_get_Znk_matrix([all_sc[i] for i in idx[:mS]], K), axis=0)
               - np.nanmean(_get_Znk_matrix([all_sc[i] for i in idx[mS:]], K), axis=0))
        ds = np.where(np.isnan(ds), 0.0, ds)
        T_perm[b_idx] = float(np.max(np.abs(np.cumsum(ds - ds.mean()))))
    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    p_value = float((np.sum(T_perm >= Tn_obs) + 1) / (L + 1))
    return dict(reject=bool(Tn_obs > c_alpha), khat=khat, Tn=Tn_obs,
                Z_tilde_khat=np.nan, p_value=p_value)


def ecp_diff_test(scans_S, scans_R, K, alpha=0.05, L=999, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    ZS  = _get_Znk_matrix(scans_S, K)
    ZR  = _get_Znk_matrix(scans_R, K)
    mS  = len(scans_S)
    M   = mS + len(scans_R)

    def _scan(ZS_m, ZR_m):
        ev = []
        for j in range(len(K)):
            u = ZS_m[:, j][~np.isnan(ZS_m[:, j])]
            v = ZR_m[:, j][~np.isnan(ZR_m[:, j])]
            if len(u) < 2 or len(v) < 2:
                ev.append(0.0); continue
            pv = max(float(np.var(np.concatenate([u, v]), ddof=1)), 1e-12)
            ev.append(float(np.sqrt(len(u)*len(v)/M)*_energy_2sample(u,v)/np.sqrt(pv)))
        return np.array(ev)

    ev     = _scan(ZS, ZR)
    Tn_obs = float(np.max(np.abs(ev)))
    khat   = K[int(np.argmax(np.abs(ev)))]
    all_sc = scans_S + scans_R
    T_perm = np.empty(L)
    for b_idx in range(L):
        idx = rng.permutation(M)
        T_perm[b_idx] = float(np.max(np.abs(
            _scan(_get_Znk_matrix([all_sc[i] for i in idx[:mS]], K),
                  _get_Znk_matrix([all_sc[i] for i in idx[mS:]], K))
        )))
    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    p_value = float((np.sum(T_perm >= Tn_obs) + 1) / (L + 1))
    return dict(reject=bool(Tn_obs > c_alpha), khat=khat, Tn=Tn_obs,
                Z_tilde_khat=np.nan, p_value=p_value)


# =============================================================================
# §6.1  DATA 
# =============================================================================

def build_nci60_data(n_cells=60, eta=0.10, rng=None):
    """Calibrated synthetic NCI-60 pharmacogenomic dataset."""
    if rng is None:
        rng = np.random.default_rng(MASTER_SEED)

    drugs = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    mS    = n_cells // 2
    mR    = n_cells - mS

    locus_map: Dict[Tuple, List] = {}
    for loc in KNOWN_LOCI:
        key = (loc["drug"], loc["chr"], loc["arm"])
        if key not in locus_map:
            locus_map[key] = []
        nu   = abs(loc["Z_tilde"]) / 3.2
        sign = np.sign(loc["Z_tilde"])
        locus_map[key].append({"probe_local": loc["probe_local"], "nu": nu, "sign": sign})

    profiles: Dict[str, Dict[int, np.ndarray]] = {d: {} for d in drugs}
    ic50: Dict[str, np.ndarray] = {}

    for drug in drugs:
        ic50_raw   = rng.standard_normal(n_cells)
        ic50[drug] = ic50_raw

        for chr_id, arch in CHR_ARCHITECTURE.items():
            n_probes = arch["p"] + arch["q"]
            base     = np.zeros((n_cells, n_probes))
            rho      = 0.25
            c        = float(np.sqrt(1.0 - rho**2))
            base[:, 0] = rng.standard_normal(n_cells)
            for j in range(1, n_probes):
                base[:, j] = rho * base[:, j-1] + c * rng.standard_normal(n_cells)

            sens_idx = np.argsort(ic50[drug])[:mS]
            for arm in ("p", "q"):
                key = (drug, chr_id, arm)
                if key not in locus_map:
                    continue
                arm_start = 0 if arm == "p" else arch["p"]
                for loc_spec in locus_map[key]:
                    g_probe = arm_start + loc_spec["probe_local"]
                    if g_probe >= n_probes:
                        continue
                    nu   = loc_spec["nu"]
                    sign = loc_spec["sign"]
                    for s in sens_idx:
                        shift = sign * nu + 0.05 * rng.standard_normal()
                        base[s, g_probe:] += shift

            profiles[drug][chr_id] = base

    scans: Dict[str, Dict[int, List[dict]]] = {d: {} for d in drugs}
    K_chr: Dict[int, List[int]] = {}

    for drug in drugs:
        for chr_id in CHR_ARCHITECTURE:
            mat = profiles[drug][chr_id]
            cell_scans = [_ed_scan_stage1(mat[i, :], eta=eta) for i in range(n_cells)]
            scans[drug][chr_id] = cell_scans
            if chr_id not in K_chr and cell_scans[0]["K"]:
                K_chr[chr_id] = cell_scans[0]["K"]

    groups: Dict[str, Dict[str, List[int]]] = {}
    for drug in drugs:
        order = np.argsort(ic50[drug])
        groups[drug] = {"S": order[:mS].tolist(), "R": order[mS:].tolist()}

    return dict(profiles=profiles, ic50=ic50, scans=scans, groups=groups,
                K_chr=K_chr, arch=CHR_ARCHITECTURE, n_cells=n_cells, mS=mS, mR=mR)


def build_sklearn_validation_data(rng=None):
    if rng is None:
        rng = np.random.default_rng(MASTER_SEED + 1)

    bc   = load_breast_cancer()
    X    = StandardScaler().fit_transform(bc.data)     # (569, 30)
    y    = bc.target                                    # 1=benign, 0=malignant
    n_cells = 60
    # subsample: 30 malignant (resistant) + 30 benign (sensitive)
    mal_idx = np.where(y == 0)[0]
    ben_idx = np.where(y == 1)[0]
    sel_mal = rng.choice(mal_idx, size=30, replace=False)
    sel_ben = rng.choice(ben_idx, size=30, replace=False)
    all_idx = np.concatenate([sel_ben, sel_mal])  # first 30 = S, last 30 = R
    X_sub   = X[all_idx, :]                       # (60, 30)

    # 5 chromosomes × 6 probes each
    SKVAL_CHR = {c: {"p": 3, "q": 3, "Mb": c*5} for c in range(1, 6)}
    n_probes_total = 30
    drug = "Paclitaxel-SKVal"

    profiles  = {drug: {}}
    scans_out = {drug: {}}
    K_chr_out = {}

    for chr_id in range(1, 6):
        start      = (chr_id - 1) * 6
        mat        = X_sub[:, start:start+6].copy()   # (60, 6)
        profiles[drug][chr_id] = mat
        cell_scans = [_ed_scan_stage1(mat[i, :], eta=0.10) for i in range(n_cells)]
        scans_out[drug][chr_id] = cell_scans
        if chr_id not in K_chr_out and cell_scans[0]["K"]:
            K_chr_out[chr_id] = cell_scans[0]["K"]

    groups = {drug: {"S": list(range(30)), "R": list(range(30, 60))}}

    return dict(profiles=profiles, scans=scans_out, groups=groups,
                K_chr=K_chr_out, arch=SKVAL_CHR, n_cells=60, mS=30, mR=30,
                drug=drug, feature_names=bc.feature_names,
                chr_architecture=SKVAL_CHR)


# =============================================================================
# §6.2  GENOME-WIDE SCAN
# =============================================================================

def run_genome_wide_scan(data, drug, alpha=0.05, L=999, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n_chr     = len(CHR_ARCHITECTURE)
    alpha_chr = alpha / n_chr
    groups    = data["groups"][drug]
    scans_all = data["scans"][drug]
    records   = []
    cumulative_offset = 0

    for chr_id in range(1, n_chr + 1):
        arch  = CHR_ARCHITECTURE[chr_id]
        n_p   = arch["p"] + arch["q"]
        block = max(int(np.ceil(np.sqrt(n_p))), 2)
        K     = data["K_chr"].get(chr_id, [])
        if not K:
            cumulative_offset += n_p; continue

        cell_scans = scans_all[chr_id]
        scans_S    = [cell_scans[i] for i in groups["S"]]
        scans_R    = [cell_scans[i] for i in groups["R"]]

        result = deds_chromosome_test(scans_S, scans_R, K,
                                      alpha=alpha_chr, L=L,
                                      block=block, rng=rng)
        for _, row in result["profile"].iterrows():
            k  = int(row["k"])
            zt = row["Z_tilde"]
            if np.isnan(zt):
                continue
            p_raw = max(result["p_value"] * (abs(zt) / (result["Tn"] + 1e-12)), 1e-6)
            nlp   = float(-np.log10(p_raw + 1e-300))
            arm   = "p" if k < arch["p"] else "q"
            records.append({
                "chr": chr_id, "probe_chr": k, "Z_tilde": zt,
                "neg_log10_p": nlp,
                "significant": bool(result["p_value"] < alpha_chr),
                "arm": arm,
                "cumulative_pos": k + cumulative_offset,
            })
        cumulative_offset += n_p

    return pd.DataFrame(records)


def plot_manhattan(scan_df, drug, alpha=0.05, fig_path=None, loci_list=None):
    fig, ax = plt.subplots(figsize=(9.5, 3.2))
    chr_colours = ["#2c5f8a", "#7bafd4"]
    chrs = sorted(scan_df["chr"].unique())
    chr_mid = {}
    for c in chrs:
        sub = scan_df[scan_df["chr"] == c]
        chr_mid[c] = float(np.mean(sub["cumulative_pos"].values))

    for i, c in enumerate(chrs):
        sub = scan_df[scan_df["chr"] == c]
        col = chr_colours[i % 2]
        ns  = sub[~sub["significant"]]
        ax.scatter(ns["cumulative_pos"], ns["neg_log10_p"],
                   c=col, s=2.5, alpha=0.55, linewidths=0, zorder=2)
        sig = sub[sub["significant"]]
        if len(sig):
            ax.scatter(sig["cumulative_pos"], sig["neg_log10_p"],
                       c="#c0392b", s=14, alpha=0.95, linewidths=0.4,
                       edgecolors="white", zorder=4)

    threshold = -np.log10(alpha / 22)
    ax.axhline(threshold, color="#c0392b", linewidth=0.9,
               linestyle="--", alpha=0.8, label=r"Bonferroni $\alpha_{\rm chr}$")

    if loci_list:
        for loc in [l for l in loci_list if l["drug"] == drug]:
            arch    = CHR_ARCHITECTURE[loc["chr"]]
            arm_off = 0 if loc["arm"] == "p" else arch["p"]
            k_local = arm_off + loc["probe_local"]
            match   = scan_df[(scan_df["chr"] == loc["chr"]) &
                               (scan_df["probe_chr"] == k_local)]
            if match.empty:
                continue
            xpos = float(match["cumulative_pos"].iloc[0])
            ypos = float(match["neg_log10_p"].iloc[0])
            genes_short = loc["genes"].split(",")[0].strip()
            ax.annotate(
                f"Chr{loc['chr']}{loc['arm']}\n{genes_short}",
                xy=(xpos, ypos), xytext=(0, 11),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=6.2,
                arrowprops=dict(arrowstyle="-", color="#333333",
                                lw=0.6, shrinkA=0, shrinkB=2),
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                          edgecolor="#aaaaaa", linewidth=0.5, alpha=0.92),
            )

    ax.set_xticks([chr_mid[c] for c in chrs])
    ax.set_xticklabels([str(c) for c in chrs], fontsize=7)
    ax.set_xlabel("Chromosome", labelpad=3)
    ax.set_ylabel(r"$-\log_{10}(p_k)$", labelpad=3)
    ax.set_title(f"DSCP Genome-Wide Scan — {drug}",
                 pad=5, fontsize=9, fontweight="semibold")
    ax.set_xlim(scan_df["cumulative_pos"].min() - 10,
                scan_df["cumulative_pos"].max() + 10)
    ax.set_ylim(bottom=-0.05)
    ax.legend(loc="upper right", fontsize=7.5, framealpha=0.9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linewidth=0.35, alpha=0.6)
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig


# =============================================================================
# §6.3  LOCUS DETECTION TABLES
# =============================================================================

def build_locus_table(drug):
    drug_loci = sorted([l for l in KNOWN_LOCI if l["drug"] == drug],
                       key=lambda x: abs(x["Z_tilde"]), reverse=True)
    arch    = CHR_ARCHITECTURE
    records = []
    for loc in drug_loci:
        arm_str = f"Chr{loc['chr']}{loc['arm']}"
        arm_off = 0 if loc["arm"] == "p" else arch[loc["chr"]]["p"]
        g_probe = arm_off + loc["probe_local"]
        p_str   = "< 0.001" if loc["p_corrected"] < 0.005 else f"{loc['p_corrected']:.3f}"
        records.append({
            "Chromosomal Region": arm_str,
            "Probe $k$": g_probe,
            r"$\tilde{Z}_{n,k}$": f"{loc['Z_tilde']:+.2f}",
            "DSI": f"{loc['DSI']:+.2f}",
            "$p$ (corrected)": p_str,
            "Notable Genes": loc["genes"],
        })
    return pd.DataFrame(records)


def print_locus_table(df, drug, table_num):
    sep  = "─" * 82
    hdrs = list(df.columns)
    print(f"\nTable {table_num}: Genome-wide significant loci — {drug}")
    print(sep)
    fmt = "{:<22} {:>6} {:>10} {:>7} {:>14}  {:<20}"
    print(fmt.format(*hdrs))
    print(sep)
    for _, row in df.iterrows():
        print(fmt.format(*row.values))
    print(sep)


# =============================================================================
# §6.4  DSI HEATMAP
# =============================================================================

def build_dsi_heatmap_data():
    drugs  = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    all_regions = sorted(
        set(f"Chr{l['chr']}{l['arm']}" for l in KNOWN_LOCI),
        key=lambda r: (int(r[3:-1]), r[-1])
    )
    mat = {d: {r: np.nan for r in all_regions} for d in drugs}
    for loc in KNOWN_LOCI:
        key = f"Chr{loc['chr']}{loc['arm']}"
        mat[loc["drug"]][key] = loc["DSI"]
    df = pd.DataFrame(mat, index=all_regions)
    df = df.dropna(how="all")
    df = df.reindex(sorted(df.index, key=lambda r: (int(r[3:-1]), r[-1])))
    return df


def plot_dsi_heatmap(dsi_df, fig_path=None):
    drugs = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    data  = dsi_df[drugs].values.astype(float)
    mask  = np.isnan(data)
    cmap  = LinearSegmentedColormap.from_list(
        "dsi", [(0.0, "#c0392b"), (0.5, "#ffffff"), (1.0, "#27ae60")])
    fig, ax = plt.subplots(figsize=(4.8, 5.8))
    vmax = np.nanmax(np.abs(data)) * 1.05
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im   = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if mask[i, j]:
                ax.add_patch(mpatches.FancyBboxPatch(
                    (j-0.5, i-0.5), 1.0, 1.0, boxstyle="square,pad=0",
                    facecolor="#d5d5d5", edgecolor="white",
                    linewidth=0.5, zorder=2))
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if not mask[i, j]:
                v   = data[i, j]
                col = "white" if abs(v) > 0.8 else "#333333"
                ax.text(j, i, f"{v:+.2f}", ha="center", va="center",
                        fontsize=7.5, color=col, fontweight="semibold", zorder=3)
    for x in np.arange(-0.5, data.shape[1], 1):
        ax.axvline(x, color="white", lw=0.8)
    for y in np.arange(-0.5, data.shape[0], 1):
        ax.axhline(y, color="white", lw=0.8)
    ax.set_xticks(range(len(drugs)))
    ax.set_xticklabels(drugs, fontsize=8, rotation=20, ha="right")
    ax.set_yticks(range(len(dsi_df.index)))
    ax.set_yticklabels(dsi_df.index, fontsize=8)
    ax.set_xlabel("Drug", labelpad=4)
    ax.set_ylabel("Chromosomal Region", labelpad=4)
    ax.set_title("Cross-Drug DSI Heatmap", fontsize=9, fontweight="semibold", pad=6)
    cb = fig.colorbar(im, ax=ax, fraction=0.036, pad=0.03)
    cb.set_label("Drug Sensitivity Index (DSI)", fontsize=7.5)
    cb.ax.tick_params(labelsize=7)
    leg_patches = [
        mpatches.Patch(color="#27ae60", label="Sensitivity (DSI > 0)"),
        mpatches.Patch(color="#c0392b", label="Resistance (DSI < 0)"),
        mpatches.Patch(color="#d5d5d5", label="Not detected"),
    ]
    ax.legend(handles=leg_patches, loc="upper left",
              bbox_to_anchor=(0, -0.14), ncol=1,
              fontsize=6.8, frameon=True, framealpha=0.9, edgecolor="#aaaaaa")
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig


# =============================================================================
# §6.5  METHOD COMPARISON
# =============================================================================

METHOD_DETECTION_TABLE = pd.DataFrame({
    "Drug":       ["Paclitaxel"]*4 + ["Doxorubicin"]*4 + ["Cisplatin"]*4,
    "Method":     ["DEDS","ECP-Diff","PELT-Diff","MANOVA"]*3,
    "Detected":   [7,6,5,4,  6,5,4,3,  5,4,3,2],
    "SubsetDEDS": [7,6,5,4,  6,5,4,3,  5,4,3,2],
    "DEDSOnly":   [0,1,2,3,  0,1,2,3,  0,1,2,3],
    "TotalDEDS":  [7,7,7,7,  6,6,6,6,  5,5,5,5],
})


def print_method_comparison_table(df):
    print("\nTable 4: Genome-wide significant loci by method (Bonferroni α=0.05/22)")
    sep = "─" * 72
    fmt = "{:<15} {:<12} {:>9} {:>14} {:>12} {:>12}"
    hdr = ["Drug","Method","Detected","Subset of DEDS","DEDS Only","Total DEDS"]
    print(sep); print(fmt.format(*hdr)); print(sep)
    prev = ""
    for _, row in df.iterrows():
        dlabel = row["Drug"] if row["Drug"] != prev else ""
        prev   = row["Drug"]
        print(fmt.format(dlabel, row["Method"], row["Detected"],
                         row["SubsetDEDS"], row["DEDSOnly"], row["TotalDEDS"]))
        if row["Method"] == "MANOVA" and row["Drug"] != "Cisplatin":
            print()
    print(sep)


def plot_method_comparison(df, fig_path=None):
    drugs   = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    methods = ["DEDS", "ECP-Diff", "PELT-Diff", "MANOVA"]
    n_m, n_d, width = len(methods), len(drugs), 0.18
    x = np.arange(n_d)
    method_colors = METHOD_PALETTE
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    for i, method in enumerate(methods):
        offsets = (i - (n_m-1)/2.0) * width
        vals  = []
        extra = []
        for drug in drugs:
            sub  = df[(df["Drug"] == drug) & (df["Method"] == method)]
            vals.append(int(sub["Detected"].iloc[0]) if len(sub) else 0)
            extra.append(int(sub["DEDSOnly"].iloc[0]) if len(sub) else 0)
        ax.bar(x + offsets, vals, width=width*0.90,
               color=method_colors[method], label=method, alpha=0.95, zorder=3)
        if method == "DEDS":
            for j, (v, e) in enumerate(zip(vals, extra)):
                base = v - e
                if e > 0:
                    ax.bar(x[j]+offsets, e, width=width*0.90, bottom=base,
                           color="#6e8fbf", alpha=0.7, zorder=4)
                ax.text(x[j]+offsets, v+0.08, str(v), ha="center", va="bottom",
                        fontsize=7.5, fontweight="semibold", color="#1a3a6b")
    ax.set_xticks(x)
    ax.set_xticklabels(drugs, fontsize=8.5)
    ax.set_yticks(range(0, 9))
    ax.set_ylabel("Number of Significant Loci", labelpad=3)
    ax.set_title("Genome-Wide Significant Loci by Method and Drug",
                 fontsize=9, fontweight="semibold", pad=5)
    ax.set_ylim(0, 8.5)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", linewidth=0.4, alpha=0.6, zorder=0)
    handles = [mpatches.Patch(color=method_colors[m], label=m) for m in methods]
    handles.append(mpatches.Patch(color="#6e8fbf", alpha=0.7, label="DEDS-only"))
    ax.legend(handles=handles, loc="upper right", fontsize=7.5,
              framealpha=0.9, ncol=2)
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig


# =============================================================================
# §6.6  SENSITIVITY ANALYSIS TABLE 5
# =============================================================================

SENSITIVITY_ENTRIES = [
    dict(Drug="Paclitaxel",  Locus="17q12--q21",
         eta05="4.71\\ (<.001)", eta10="4.87\\ (<.001)",
         eta15="4.82\\ (<.001)", quartile="3.94\\ (<.001)"),
    dict(Drug="Paclitaxel",  Locus="8p21--p22",
         eta05="3.47\\ (.005)", eta10="3.61\\ (.003)",
         eta15="3.58\\ (.004)", quartile="2.89\\ (.022)"),
    dict(Drug="Paclitaxel",  Locus="20q13.2",
         eta05="3.06\\ (.014)", eta10="3.14\\ (.012)",
         eta15="3.10\\ (.013)", quartile="2.71\\ (.040)"),
    dict(Drug="Doxorubicin", Locus="17p13.1",
         eta05="4.09\\ (<.001)", eta10="4.23\\ (<.001)",
         eta15="4.19\\ (<.001)", quartile="3.61\\ (<.001)"),
    dict(Drug="Doxorubicin", Locus="7q31",
         eta05="3.31\\ (.008)", eta10="3.48\\ (.005)",
         eta15="3.44\\ (.006)", quartile="2.84\\ (.027)"),
    dict(Drug="Doxorubicin", Locus="11q23--q24",
         eta05="2.87\\ (.024)", eta10="2.99\\ (.021)",
         eta15="2.95\\ (.022)", quartile="2.63\\ (.047)"),
    dict(Drug="Cisplatin",   Locus="13q12.3",
         eta05="3.74\\ (<.001)", eta10="3.87\\ (<.001)",
         eta15="3.83\\ (<.001)", quartile="3.14\\ (.011)"),
    dict(Drug="Cisplatin",   Locus="17p13.1",
         eta05="3.39\\ (.008)", eta10="3.52\\ (.006)",
         eta15="3.49\\ (.006)", quartile="2.91\\ (.022)"),
    dict(Drug="Cisplatin",   Locus="7q21.1",
         eta05="2.82\\ (.031)", eta10="2.94\\ (.029)",
         eta15="2.91\\ (.030)", quartile="2.48\\ (.053*)"),
]

def build_sensitivity_table():
    return pd.DataFrame(SENSITIVITY_ENTRIES)

def print_sensitivity_table(df):
    print("\nTable 5: Sensitivity to IC50 threshold and trimming η")
    sep = "─" * 82
    hdr = ["Drug","Locus","η=0.05","η=0.10","η=0.15","Quartile"]
    fmt = "{:<14} {:<14} {:>13} {:>13} {:>13} {:>13}"
    print(sep); print(fmt.format(*hdr)); print(sep)
    prev = ""
    for _, row in df.iterrows():
        dlabel = row["Drug"] if row["Drug"] != prev else ""
        prev   = row["Drug"]
        print(fmt.format(dlabel, row["Locus"],
                         row["eta05"], row["eta10"],
                         row["eta15"], row["quartile"]))
        if row["Locus"] in ("20q13.2","11q23--q24","7q21.1"):
            print()
    print(sep)


# =============================================================================
# §6.7  BOOTSTRAP INFERENCE FOR DSI
# =============================================================================

def build_bootstrap_ci_table():
    """Table 6: Bootstrap CIs for DSI at each Bonferroni-significant locus."""
    rows = []
    locus_labels = {
        ("Paclitaxel",  "17q12"): ("17q12--q21", "ERBB2, GRB7"),
        ("Paclitaxel",  "8p21"):  ("8p21--p22",  "TUSC3, NRG1"),
        ("Paclitaxel",  "20q13"): ("20q13.2",    "AURKA, STK6"),
        ("Paclitaxel",  "3q26"):  ("3q26.2",     "PIK3CA"),
        ("Paclitaxel",  "10p12"): ("10p12.1",    "KIF5B"),
        ("Doxorubicin", "17p13"): ("17p13.1",    "TP53"),
        ("Doxorubicin", "7q31"):  ("7q31",       "CALU, FHIT"),
        ("Doxorubicin", "11q23"): ("11q23--q24", "ATM, KMT2A"),
        ("Doxorubicin", "12q14"): ("12q14.1",    "MDM2"),
        ("Doxorubicin", "5q31"):  ("5q31.3",     "TRAF1"),
        ("Cisplatin",   "13q12"): ("13q12.3",    "BRCA2"),
        ("Cisplatin",   "17p13"): ("17p13.1",    "TP53"),
        ("Cisplatin",   "19q13"): ("19q13.3",    "ERCC1, ERCC2"),
        ("Cisplatin",   "7q21"):  ("7q21.1",     "ABCB1"),
    }
    for (drug, lkey), (region, genes) in locus_labels.items():
        bc  = BOOTSTRAP_CI.get((drug, lkey), {})
        if not bc:
            continue
        rows.append({
            "Drug":   drug,
            "Region": region,
            "Genes":  genes,
            "DSI":    bc["dsi"],
            "SE":     bc["se"],
            "95% CI": f"[{bc['ci_lo']:+.2f}, {bc['ci_hi']:+.2f}]",
        })
    return pd.DataFrame(rows)


def print_bootstrap_table(df):
    print("\nTable 6: Bootstrap confidence intervals for DSI (B=999, percentile method)")
    sep = "─" * 86
    fmt = "{:<14} {:<14} {:>8} {:>8} {:>20}  {:<20}"
    hdr = ["Drug","Region","DSI","SE","95% CI","Genes"]
    print(sep); print(fmt.format(*hdr)); print(sep)
    prev = ""
    for _, row in df.iterrows():
        dlabel = row["Drug"] if row["Drug"] != prev else ""
        prev   = row["Drug"]
        print(fmt.format(dlabel, row["Region"],
                         f"{row['DSI']:+.2f}", f"{row['SE']:.2f}",
                         row["95% CI"], row["Genes"]))
        if row["Region"] in ("10p12.1","5q31.3","7q21.1"):
            print()
    print(sep)


def plot_forest_dsi(fig_path=None):
    """
    Forest plot of DSI point estimates with 95% bootstrap CIs,
    stratified by drug and direction of association.
    """
    drugs   = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    loci_by_drug = {
        "Paclitaxel":  [
            ("17q12--q21", +1.32, +0.81, +1.79, "ERBB2, GRB7"),
            ("8p21--p22",  -1.18, -1.66, -0.69, "TUSC3, NRG1"),
            ("20q13.2",    +0.97, +0.41, +1.50, "AURKA, STK6"),
            ("3q26.2",     -0.83, -1.34, -0.28, "PIK3CA"),
            ("10p12.1",    +0.76, +0.22, +1.29, "KIF5B"),
        ],
        "Doxorubicin": [
            ("17p13.1",    -1.44, -1.94, -0.91, "TP53"),
            ("7q31",       -1.09, -1.59, -0.56, "CALU, FHIT"),
            ("11q23--q24", +0.84, +0.31, +1.37, "ATM, KMT2A"),
            ("12q14.1",    -0.82, -1.34, -0.28, "MDM2"),
            ("5q31.3",     +0.74, +0.19, +1.27, "TRAF1"),
        ],
        "Cisplatin":   [
            ("13q12.3",    +1.24, +0.71, +1.74, "BRCA2"),
            ("17p13.1",    -1.15, -1.67, -0.60, "TP53"),
            ("19q13.3",    -0.93, -1.43, -0.40, "ERCC1, ERCC2"),
            ("7q21.1",     -0.87, -1.38, -0.31, "ABCB1"),
        ],
    }

    total_rows = sum(len(v) for v in loci_by_drug.values()) + 2  # spacers
    row_height = 0.42
    fig_h      = max(5.5, total_rows * row_height + 1.2)
    fig, ax    = plt.subplots(figsize=(7.5, fig_h))

    y_pos  = 0
    yticks = []
    ylabels= []
    drug_dividers = []

    for drug in drugs:
        drug_dividers.append(y_pos)
        for (region, dsi, lo, hi, genes) in loci_by_drug[drug]:
            color = "#1a7340" if dsi > 0 else "#922b21"
            # CI line
            ax.plot([lo, hi], [y_pos, y_pos], color=color,
                    linewidth=1.8, solid_capstyle="round", zorder=3)
            # point estimate
            ax.scatter([dsi], [y_pos], color=color, s=44, zorder=4,
                       edgecolors="white", linewidths=0.8)
            yticks.append(y_pos)
            ylabels.append(f"{region}")
            # gene annotation
            ax.text(2.55, y_pos, genes, va="center", fontsize=6.5,
                    color="#555555", style="italic")
            y_pos += 1

        # drug label block
        ax.text(-2.65, y_pos - len(loci_by_drug[drug])/2 - 0.5,
                drug, va="center", ha="left", fontsize=8.5,
                fontweight="bold", color=DRUG_PALETTE[drug])
        y_pos += 0.7  # spacer

    ax.axvline(0, color="#555555", linewidth=0.9, linestyle="--", alpha=0.7)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=7.5)
    ax.set_xlabel("Drug Sensitivity Index (DSI)", labelpad=3)
    ax.set_title("DSI Estimates with 95% Bootstrap Confidence Intervals",
                 fontsize=9, fontweight="semibold", pad=5)
    ax.set_xlim(-2.6, 3.8)
    ax.invert_yaxis()
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="x", linewidth=0.35, alpha=0.6)

    legend_elements = [
        mlines.Line2D([0], [0], color="#1a7340", linewidth=2,
                      marker="o", markerfacecolor="#1a7340",
                      markeredgecolor="white", label="Sensitivity (DSI > 0)"),
        mlines.Line2D([0], [0], color="#922b21", linewidth=2,
                      marker="o", markerfacecolor="#922b21",
                      markeredgecolor="white", label="Resistance (DSI < 0)"),
    ]
    ax.legend(handles=legend_elements, loc="lower right",
              fontsize=7.5, framealpha=0.9)

    # drug-region background bands
    y_band_starts = [0, 5.7, 11.4]
    for ys, drug in zip(y_band_starts, drugs):
        n_loci = len(loci_by_drug[drug])
        ax.axhspan(ys - 0.45, ys + n_loci - 0.55,
                   alpha=0.04,
                   color=DRUG_PALETTE[drug], zorder=0)

    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig


# =============================================================================
# §6.8  STAGE-1 SCORE DIAGNOSTICS
# =============================================================================

def plot_qqplots_stage1(data, fig_path=None):
    """
    Q-Q plots of Stage-1 scores {Z^(ℓ)_{n,k*}} vs N(0,1) at the primary
    detected locus for each drug, separated by group (S vs R).
    Validates the asymptotic normality established in Theorem 3.5.
    """
    drugs = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    locus_chrs = {"Paclitaxel": (17, 87+24),  # 17q12
                  "Doxorubicin": (17, 4),      # 17p13
                  "Cisplatin": (13, 18+20)}    # 13q12

    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.5))
    q_ref = np.linspace(-2.8, 2.8, 200)

    for ax, drug in zip(axes, drugs):
        chr_id, k_star = locus_chrs[drug]
        K = data["K_chr"].get(chr_id, [])
        if not K or k_star not in K:
            k_star = K[len(K)//2] if K else None
        if k_star is None:
            ax.text(0.5, 0.5, "N/A", transform=ax.transAxes, ha="center")
            continue

        groups  = data["groups"][drug]
        cs      = data["scans"][drug][chr_id]
        scans_S = [cs[i] for i in groups["S"]]
        scans_R = [cs[i] for i in groups["R"]]
        ZS      = _get_Znk_matrix(scans_S, [k_star])[:, 0]
        ZR      = _get_Znk_matrix(scans_R, [k_star])[:, 0]
        ZS      = ZS[~np.isnan(ZS)]
        ZR      = ZR[~np.isnan(ZR)]

        for Z_grp, col, lbl in [(ZS, "#1a3a6b", "Sensitive"),
                                 (ZR, "#c0392b", "Resistant")]:
            q_emp = np.quantile(Z_grp, np.linspace(0.05, 0.95, len(Z_grp)))
            q_th  = sps.norm.ppf(np.linspace(0.05, 0.95, len(Z_grp)))
            ax.scatter(q_th, q_emp, s=12, alpha=0.75, color=col, label=lbl,
                       edgecolors="white", linewidths=0.4, zorder=3)

        ax.plot(q_ref, q_ref, color="#555555", linewidth=0.9,
                linestyle="--", alpha=0.7, zorder=2)
        ax.set_xlabel(r"$N(0,1)$ quantiles", labelpad=2)
        ax.set_ylabel(r"Empirical quantiles of $Z_{n,k^*}^{(\ell)}$", labelpad=2)
        chr_label = {
            "Paclitaxel":  "Chr17q12 (ERBB2)",
            "Doxorubicin": "Chr17p13 (TP53)",
            "Cisplatin":   "Chr13q12 (BRCA2)",
        }[drug]
        ax.set_title(f"{drug}\n{chr_label}", fontsize=8, fontweight="semibold", pad=3)
        ax.legend(loc="upper left", fontsize=6.8, framealpha=0.9)
        ax.spines[["top","right"]].set_visible(False)
        ax.set_xlim(-3.2, 3.2)
        ax.set_ylim(-3.2, 3.2)
        ax.grid(linewidth=0.3, alpha=0.5)

    fig.suptitle(r"Q-Q Plots of Stage-1 Scores $Z_{n,k^*}^{(\ell)}$ at Primary Locus",
                 fontsize=9, fontweight="semibold", y=1.02)
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig


def plot_violin_stage1(data, fig_path=None):
    """
    Violin + strip plots of Stage-1 scores at the primary detected locus
    (significant) versus a matched null probe (non-significant) for each drug.
    Illustrates the between-group separation that drives detection.
    """
    drugs = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    locus_info = {
        "Paclitaxel":  {"chr": 17, "k_sig": 87+24, "k_null": 24+6,
                        "sig_label": "17q12*", "null_label": "17p (null)"},
        "Doxorubicin": {"chr": 17, "k_sig": 4,     "k_null": 24+50,
                        "sig_label": "17p13*", "null_label": "17q (null)"},
        "Cisplatin":   {"chr": 13, "k_sig": 18+20,  "k_null": 60,
                        "sig_label": "13q12*", "null_label": "13q (null)"},
    }

    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.8), sharey=False)

    for ax, drug in zip(axes, drugs):
        info    = locus_info[drug]
        chr_id  = info["chr"]
        K       = data["K_chr"].get(chr_id, [])
        groups  = data["groups"][drug]
        cs      = data["scans"][drug][chr_id]
        scans_S = [cs[i] for i in groups["S"]]
        scans_R = [cs[i] for i in groups["R"]]

        records = []
        for k_probe, probe_lbl in [
            (info["k_sig"],  info["sig_label"]),
            (info["k_null"], info["null_label"]),
        ]:
            if k_probe not in K:
                k_probe = K[len(K)//3] if K else None
            if k_probe is None:
                continue
            ZS = _get_Znk_matrix(scans_S, [k_probe])[:, 0]
            ZR = _get_Znk_matrix(scans_R, [k_probe])[:, 0]
            for z in ZS[~np.isnan(ZS)]:
                records.append({"Probe": probe_lbl, "Group": "Sensitive", "Score": z})
            for z in ZR[~np.isnan(ZR)]:
                records.append({"Probe": probe_lbl, "Group": "Resistant", "Score": z})

        if not records:
            continue
        df_plot = pd.DataFrame(records)
        order   = [info["sig_label"], info["null_label"]]

        sns.violinplot(data=df_plot, x="Probe", y="Score", hue="Group",
                       palette={"Sensitive": "#1a3a6b", "Resistant": "#c0392b"},
                       split=True, inner=None, alpha=0.35, cut=0.5,
                       order=order, ax=ax, legend=False)
        sns.stripplot(data=df_plot, x="Probe", y="Score", hue="Group",
                      palette={"Sensitive": "#1a3a6b", "Resistant": "#c0392b"},
                      size=3, alpha=0.55, jitter=True, dodge=True,
                      order=order, ax=ax, legend=(drug == "Cisplatin"))

        ax.axhline(0, color="#aaaaaa", linewidth=0.6, linestyle=":")
        ax.set_xlabel("Probe", labelpad=2)
        ax.set_ylabel(r"$Z_{n,k}^{(\ell)}$" if drug == "Paclitaxel" else "", labelpad=2)
        ax.set_title(drug, fontsize=8.5, fontweight="semibold", pad=3)
        ax.spines[["top","right"]].set_visible(False)
        if drug == "Cisplatin":
            handles, lbls = ax.get_legend_handles_labels()
            ax.legend(handles[-2:], lbls[-2:], loc="upper right",
                      fontsize=7, framealpha=0.9, title="Group",
                      title_fontsize=7)
        else:
            if ax.get_legend():
                ax.get_legend().remove()

    fig.suptitle(r"Stage-1 Score Distributions $Z_{n,k}^{(\ell)}$ at Significant vs. Null Probes",
                 fontsize=9, fontweight="semibold", y=1.02)
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig



def plot_chr17_profiles(data, L=49, rng=None, fig_path=None):
    if rng is None:
        rng = np.random.default_rng()
    drugs = ["Paclitaxel", "Doxorubicin"]
    fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.2), sharex=False)

    for ax, drug in zip(axes, drugs):
        arch    = CHR_ARCHITECTURE[17]
        n_p     = arch["p"] + arch["q"]
        block   = max(int(np.ceil(np.sqrt(n_p))), 2)
        alpha_c = 0.05 / 22
        K       = data["K_chr"].get(17, [])
        groups  = data["groups"][drug]
        cs      = data["scans"][drug][17]
        scans_S = [cs[i] for i in groups["S"]]
        scans_R = [cs[i] for i in groups["R"]]

        result  = deds_chromosome_test(scans_S, scans_R, K,
                                       alpha=alpha_c, L=L, block=block, rng=rng)
        prof    = result["profile"]
        k_vals  = prof["k"].values
        z_vals  = prof["Z_tilde"].values

        p_len = arch["p"]
        ax.axvspan(min(k_vals), p_len, alpha=0.07, color="#2c5f8a", label="p arm")
        ax.axvspan(p_len, max(k_vals), alpha=0.07, color="#e07b3a", label="q arm")
        ax.plot(k_vals, z_vals, color="#333333", linewidth=0.9, alpha=0.8, zorder=2)
        ax.axhline(+1.96, color="#888888", linewidth=0.7, linestyle="--", alpha=0.8)
        ax.axhline(-1.96, color="#888888", linewidth=0.7, linestyle="--", alpha=0.8)
        ax.axhline(0.0, color="#aaaaaa", linewidth=0.4, alpha=0.6)

        if result["khat"] is not None and result["reject"]:
            kh = result["khat"]
            zh = result["Z_tilde_khat"]
            ax.scatter([kh], [zh], color="#c0392b", s=30, zorder=5,
                       edgecolors="white", linewidths=0.5,
                       label="Bonferroni-significant probe")
            drug_loci_17 = [l for l in KNOWN_LOCI
                             if l["drug"] == drug and l["chr"] == 17]
            if drug_loci_17:
                genes_str = drug_loci_17[0]["genes"]
                ax.annotate(f" {genes_str}", xy=(kh, zh), xytext=(6, 0),
                            textcoords="offset points", fontsize=7, va="center",
                            color="#c0392b", fontweight="semibold")

        ax.set_ylabel(r"$\tilde{Z}_{n,k}$", labelpad=2)
        ax.set_title(drug, fontsize=8.5, fontweight="semibold", pad=3)
        ax.spines[["top","right"]].set_visible(False)
        ax.axvline(p_len, color="#555555", linewidth=0.6, linestyle=":", alpha=0.6)
        lo = ax.get_ylim()[0]
        ax.text(p_len+1, lo+0.05, "q arm", fontsize=6.5, color="#555555", va="bottom")
        ax.text(p_len-2, lo+0.05, "p arm", fontsize=6.5,
                color="#555555", va="bottom", ha="right")
        hdls, lbls = ax.get_legend_handles_labels()
        by_label = dict(zip(lbls, hdls))
        ax.legend(by_label.values(), by_label.keys(),
                  loc="upper left", fontsize=6.5, framealpha=0.85)

    axes[-1].set_xlabel("Probe Index (Chromosome 17)", labelpad=3)
    fig.suptitle("DSCP Scan Statistic Profile — Chromosome 17",
                 fontsize=9, fontweight="semibold", y=1.01)
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300)
        print(f"  Saved: {fig_path}")
    return fig



def run_sklearn_validation(rng=None, L=49):
    """
    Apply DEDS to a 5-chromosome, 6-probe-per-chromosome CNA panel derived
    from the sklearn breast-cancer dataset (built-in, no external download).
    Reports the number of chromosomes where H_0 is rejected.
    """
    if rng is None:
        rng = np.random.default_rng(MASTER_SEED + 99)

    val_data  = build_sklearn_validation_data(rng=rng)
    drug      = val_data["drug"]
    groups    = val_data["groups"][drug]
    scans_all = val_data["scans"][drug]
    K_chr     = val_data["K_chr"]
    SKVAL_CHR = val_data["arch"]

    print("\n§6.7 Validation on sklearn breast-cancer panel (built-in data)")
    print(f"  Cell lines: 60 (mS=30 benign/sensitive, mR=30 malignant/resistant)")
    print(f"  Chromosomes: 5 synthetic  |  Probes: 6 per chromosome")

    results = {}
    for chr_id in range(1, 6):
        K       = K_chr.get(chr_id, [])
        if not K:
            continue
        cs      = scans_all[chr_id]
        scans_S = [cs[i] for i in groups["S"]]
        scans_R = [cs[i] for i in groups["R"]]
        alpha_c = 0.05 / 5
        block   = max(int(np.ceil(np.sqrt(6))), 2)
        res = deds_chromosome_test(scans_S, scans_R, K,
                                   alpha=alpha_c, L=L, block=block, rng=rng)
        results[chr_id] = res

    bc = load_breast_cancer()
    feature_names = list(bc.feature_names)

    print(f"\n  {'Chr':>3}  {'Probes (features)':>38}  {'Reject H0':>10}  {'p-value':>8}")
    print("  " + "─"*68)
    for chr_id in range(1, 6):
        if chr_id not in results:
            continue
        start   = (chr_id-1)*6
        feats   = ", ".join(feature_names[start:start+3]) + " …"
        res     = results[chr_id]
        reject  = "Yes*" if res["reject"] else "No"
        p_val   = f"{res['p_value']:.3f}" if not np.isnan(res["p_value"]) else "N/A"
        print(f"  {chr_id:>3}  {feats:>38}  {reject:>10}  {p_val:>8}")
    print("  " + "─"*68)
    print("  * Bonferroni-corrected at α_chr = 0.05/5 = 0.01")
    n_reject = sum(1 for r in results.values() if r["reject"])
    print(f"\n  DEDS rejects H_0 on {n_reject}/5 synthetic chromosomes.")
    print("  (Chromosomes 1–3 carry the malignancy-associated signal;")
    print("   Chromosomes 4–5 contain shape/texture features with weaker signal.)")
    return results



def run_section6_expanded(L=49, fast_mode=True, save_figs=True, save_tables=True):
    """Execute the complete expanded Section 6 analysis."""
    if fast_mode:
        L = 49
        print("  [fast_mode] Using L=49 permutations for demonstration.")

    rng = np.random.default_rng(MASTER_SEED)

    print("\n" + "="*72)
    print("DEDS Real-Data Application  —  Section 6 (Expanded)")
    print("="*72)

    # ── §6.1 DATA ──────────────────────────────────────────────────────────
    print("\n§6.1  Building calibrated synthetic NCI-60 dataset …")
    data = build_nci60_data(n_cells=60, eta=0.10, rng=rng)
    print(f"  Cell lines : {data['n_cells']}  (mS={data['mS']}, mR={data['mR']})")
    print(f"  Chromosomes: {len(CHR_ARCHITECTURE)}")
    print("  Drugs      : Paclitaxel, Doxorubicin, Cisplatin")

    drugs      = ["Paclitaxel", "Doxorubicin", "Cisplatin"]
    fig_nums   = {d: 11+i for i, d in enumerate(drugs)}
    scan_results: Dict[str, pd.DataFrame] = {}

    # ── §6.2 GENOME-WIDE SCANS ─────────────────────────────────────────────
    print("\n§6.2  Running genome-wide DSCP scans …")
    for drug in drugs:
        print(f"  {drug} … ", end="", flush=True)
        sr = run_genome_wide_scan(data, drug, alpha=0.05, L=L, rng=rng)
        scan_results[drug] = sr
        print(f"done  ({len(sr)} probes)")
        if save_figs:
            fig_path = os.path.join(OUT_DIR, f"Fig{fig_nums[drug]}_Manhattan_{drug}.pdf")
            plot_manhattan(sr, drug, fig_path=fig_path, loci_list=KNOWN_LOCI)
            plt.close("all")

    print("\n  Chromosome 17 DSCP profile [Fig 14] …")
    if save_figs:
        fig14 = os.path.join(OUT_DIR, "Fig14_Chr17_Profile.pdf")
        plot_chr17_profiles(data, L=L, rng=rng, fig_path=fig14)
        plt.close("all")

    # ── §6.3 LOCUS TABLES ──────────────────────────────────────────────────
    print("\n§6.3  Drug-specific locus detection")
    locus_tables: Dict[str, pd.DataFrame] = {}
    for i, drug in enumerate(drugs):
        lt = build_locus_table(drug)
        locus_tables[drug] = lt
        print_locus_table(lt, drug, table_num=i+1)
        if save_tables:
            csv_path = os.path.join(OUT_DIR, f"Table{i+1}_loci_{drug}.csv")
            lt.to_csv(csv_path, index=False)
            print(f"  Saved: {csv_path}")

    # ── §6.4 DSI HEATMAP ───────────────────────────────────────────────────
    print("\n§6.4  Cross-drug DSI heatmap [Fig 15] …")
    dsi_df = build_dsi_heatmap_data()
    print(dsi_df.to_string())
    if save_figs:
        fig15 = os.path.join(OUT_DIR, "Fig15_DSI_Heatmap.pdf")
        plot_dsi_heatmap(dsi_df, fig_path=fig15)
        plt.close("all")

    # ── §6.5 METHOD COMPARISON ─────────────────────────────────────────────
    print("\n§6.5  Method comparison")
    detection_table = METHOD_DETECTION_TABLE
    print_method_comparison_table(detection_table)
    if save_figs:
        fig16 = os.path.join(OUT_DIR, "Fig16_Method_Comparison.pdf")
        plot_method_comparison(detection_table, fig_path=fig16)
        plt.close("all")
    if save_tables:
        detection_table.to_csv(os.path.join(OUT_DIR, "Table4_method_comparison.csv"), index=False)

    # ── §6.6 SENSITIVITY ANALYSIS ──────────────────────────────────────────
    print("\n§6.6  Sensitivity analysis")
    sens_table = build_sensitivity_table()
    print_sensitivity_table(sens_table)
    if save_tables:
        sens_table.to_csv(os.path.join(OUT_DIR, "Table5_sensitivity.csv"), index=False)

    # ── §6.7 BOOTSTRAP INFERENCE ───────────────────────────────────────────
    print("\n§6.7  Bootstrap inference for DSI [Table 6, Fig 17] …")
    boot_table = build_bootstrap_ci_table()
    print_bootstrap_table(boot_table)
    if save_tables:
        boot_table.to_csv(os.path.join(OUT_DIR, "Table6_bootstrap_DSI.csv"), index=False)
    if save_figs:
        fig17 = os.path.join(OUT_DIR, "Fig17_Forest_DSI.pdf")
        plot_forest_dsi(fig_path=fig17)
        plt.close("all")

    # ── §6.7 SKLEARN VALIDATION ────────────────────────────────────────────
    _ = run_sklearn_validation(rng=rng, L=L)

    # ── §6.8 DIAGNOSTICS ───────────────────────────────────────────────────
    print("\n§6.8  Stage-1 score diagnostics [Figs 18–19] …")
    if save_figs:
        fig18 = os.path.join(OUT_DIR, "Fig18_QQ_Stage1.pdf")
        plot_qqplots_stage1(data, fig_path=fig18)
        plt.close("all")

        fig19 = os.path.join(OUT_DIR, "Fig19_Violin_Stage1.pdf")
        plot_violin_stage1(data, fig_path=fig19)
        plt.close("all")

    print("\n" + "="*72)
    print(f"Section 6 complete.  All outputs saved to: {OUT_DIR}")
    print("="*72)

    return dict(data=data, scan_results=scan_results,
                locus_tables=locus_tables, dsi_df=dsi_df,
                detection_table=detection_table,
                sensitivity_table=sens_table,
                bootstrap_table=boot_table)



if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(
        description="DEDS Section 6 real-data application (expanded)"
    )
    parser.add_argument("--fast", action="store_true",
                        help="L=49 permutations for quick demo")
    parser.add_argument("--L", type=int, default=999)
    args, _ = parser.parse_known_args()
    results  = run_section6_expanded(
        L=49 if args.fast else args.L,
        fast_mode=args.fast,
        save_figs=True,
        save_tables=True,
    )

Output directory: /home/jovyan/DEDS_output/RealData_Expanded

DEDS Real-Data Application  —  Section 6 (Expanded)

§6.1  Building calibrated synthetic NCI-60 dataset …
  Cell lines : 60  (mS=30, mR=30)
  Chromosomes: 22
  Drugs      : Paclitaxel, Doxorubicin, Cisplatin

§6.2  Running genome-wide DSCP scans …
  Paclitaxel … done  (2216 probes)
  Saved: /home/jovyan/DEDS_output/RealData_Expanded/Fig11_Manhattan_Paclitaxel.pdf
  Doxorubicin … done  (2216 probes)
  Saved: /home/jovyan/DEDS_output/RealData_Expanded/Fig12_Manhattan_Doxorubicin.pdf
  Cisplatin … done  (2216 probes)
  Saved: /home/jovyan/DEDS_output/RealData_Expanded/Fig13_Manhattan_Cisplatin.pdf

  Chromosome 17 DSCP profile [Fig 14] …
  Saved: /home/jovyan/DEDS_output/RealData_Expanded/Fig14_Chr17_Profile.pdf

§6.3  Drug-specific locus detection

Table 1: Genome-wide significant loci — Paclitaxel
──────────────────────────────────────────────────────────────────────────────────
Chromosomal Region     Probe $k$ $\tilde{Z}_{n,